# Homework 5 - API Practice

## PART 1: Last.fm API

### a. Load the API key and .env

This block imports the libraries needed to make API requests and organize the results into a DataFrame. It loads the Last.fm API key from `.env` and defines the API endpoint and artist name for subsequent requests.

In [1]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv

# load environment variables
load_dotenv()

# set up Last.fm API key and API URL
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
url = "https://ws.audioscrobbler.com/2.0/"

# set up artist name for API queries
artist_name = "Jay Chou"

### b. Request Jay Chou’s five most popular albums

This block calls Last.fm’s `artist.getTopAlbums` method using Jay Chou’s name. The `limit` parameter requests five results, and `format="json"` makes the response easier to process in Python. Here, the returned albums are ranked by Last.fm popularity.

In [2]:
# get top albums for the specified artist from Last.fm API
params = {
    "method": "artist.getTopAlbums",
    "artist": artist_name,
    "api_key": LASTFM_API_KEY,
    "format": "json",
    "limit": 5,
}

# make the request to get top albums for the artist
response = requests.get(url, params=params, timeout=30)
response.status_code

200

This block converts the response into a Python dictionary and extracts the album names from the nested results.

In [3]:
# set payload from the API response
payload = response.json()

# extract album names from the payload
albums = [album["name"] for album in payload["topalbums"]["album"]]
albums

["November's Chopin", '七里香', '我很忙', '葉惠美', '魔杰座']

### c. Collect track information and listening statistics

This block calls `album.getInfo` for each selected album to retrieve its tracklist. Since the naming bug in Last.fm, I queries “11月的蕭邦” when the selected album is “November’s Chopin” to obtain the Chinese track names. 

For each song, it calls `track.getInfo` to retrieve the play count and listener count. Within《我很忙》, since the naming bug in Last.fm, I changes “清花瓷” to “青花瓷” and “距离” to “我不配” before making the request. This ensures that the output name and listening statistics refer to the same selected Last.fm record.

Each record contains the album, song name, track number, duration in seconds, play count, and listener count. Numeric fields are converted to integers so they can be used in later calculations.

In [4]:
rows = []

# iterate over each selected album and fetch track information from API
for album in albums:
    # since the lastfm API bug, we need to correct the album name for the specific case
    lookup_album = "11月的蕭邦" if album == "November's Chopin" else album

    # set up query parameters for API request to get album info
    params = {
        "method": "album.getInfo",
        "artist": "周杰倫",
        "album": lookup_album,
        "api_key": LASTFM_API_KEY,
        "format": "json"
    }

    # make the request to API to get album info
    response = requests.get(url, params=params)
    payload = response.json()

    # extract album info and track list from the response
    info = payload["album"]
    tracks = info.get("tracks", {}).get("track", [])

    # ensure tracks is a list even if there's only one track
    if not isinstance(tracks, list):
        tracks = [tracks]

    # iterate over each track and fetch detailed track info
    for track in tracks:
        # correct the song name for specific cases due to Last.fm API bug
        if lookup_album == "我很忙":
            song_name = {"清花瓷": "青花瓷","距离": "我不配"}.get(track["name"], track["name"])
        else:
            song_name = track["name"]

        # set up query parameters for API request to get track info
        params = {
                "method": "track.getInfo",
                "artist": info["artist"],
                "track": song_name,
                "api_key": LASTFM_API_KEY,
                "format": "json"
            }

        # make the request to API to get track info
        response = requests.get(url, params=params)
        payload = response.json()
        track_info = payload["track"]

        # append track information to rows list
        rows.append({
            "album": lookup_album,
            "song_name": song_name,
            "track_number": int(track["@attr"]["rank"]),
            "duration_seconds": int(track.get("duration")),
            "track_playcount": int(track_info["playcount"]),
            "track_listeners": int(track_info["listeners"])
        })

This block converts the collected records into a DataFrame. Displaying the first ten rows provides a quick check of the columns and returned values.

In [5]:
# create a DataFrame from the collected track information and display the first 10 rows
lastfm = pd.DataFrame(rows)
lastfm.head(10)

,album,song_name,track_number,duration_seconds,track_playcount,track_listeners
0,11月的蕭邦,夜曲,1,229,172050,16550
1,11月的蕭邦,藍色風暴,2,284,27143,4345
2,11月的蕭邦,髮如雪,3,299,101027,10018
3,11月的蕭邦,黑色毛衣,4,240,85914,10311
4,11月的蕭邦,四面楚歌,5,244,49198,6837
5,11月的蕭邦,楓,6,278,103963,10527
6,11月的蕭邦,浪漫手機,7,237,51594,6540
7,11月的蕭邦,逆鱗,8,231,25084,4165
8,11月的蕭邦,麥芽糖,9,258,39897,5747
9,11月的蕭邦,珊瑚海,10,254,121419,13082


This block saves the DataFrame to csv file for the Genius lookup and analysis steps.

In [6]:
# save the collected track information to a csv file
lastfm.to_csv("../data/lastfm.csv", index=False)